In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import shapiro
import io
import os
import csv
import fnmatch
import datetime
from google.colab import data_table
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
from sklearn.metrics import (confusion_matrix, roc_curve, roc_auc_score,
                             accuracy_score, recall_score, precision_score,
                             f1_score, classification_report)

from sklearn.tree import DecisionTreeClassifier
from sklearn import tree # for plotting the decision tree
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression

data_table.enable_dataframe_formatter() # Shows dataframes in an Excel-like manner

# Renaming columns

The updated files (high frequency data for GHI, DHI, BNI instead of hourly that were provided in Phase 2) had the column headers in different order and with slightly different names. So I just make sure that I rename the headers to match the code from Phase 2.

In [ ]:
path = '../data/TH/' # Path where project files are located
input = 'input/drifted-nlb/' # Folder where input files (merged datasets) are located
results = 'results/' # Folder where results are saved

filenames = [f for f in os.listdir(path + input) if fnmatch.fnmatch(f, '*.csv')]
filenames = sorted(filenames)

# Define a dictionary to map old column names to new column names
column_map = {
  'TAGNAME': 'Tagname',
  'TIME': 'Date',
  'SUNANGLE': 'Sunangle',
  'VALUE': 'Value',
  'Temp 2m': 'Temp2m',
  'Dewpoint 2m': 'Dewpoint2m',
}

# For each dataset, do the following
for f in filenames:
  print('======================== Dataset: ' + str(f) + '========================')
  data = pd.read_csv(path + input + f)
  # Rename the columns
  data = data.rename(columns=column_map)
  # Define the new order of columns
  columns_ordered = ['Tagname', 'Date', 'Sunangle', 'Temp2m', 'Dewpoint2m', 'Pressure', 'Precipitation', 'GHI', 'DHI', 'BNI', 'Value']
  # Reorder the columns
  data = data.reindex(columns=columns_ordered)
  data.to_csv(path + results + f, index=False)

In [ ]:
path = '../data/NLB/'

feature = 'DHI'

# Assuming that 'data.csv' is your data file.
df = pd.read_csv(path + 'nlb_merged_utc_cl.csv', sep='\,') # Change this to the path of your csv file.

# Convert the 'Date' column to datetime format
df['Date'] = pd.to_datetime(df['Date'])

# Calculate z-score to detect outliers for the 'Temp2m' column.
df['z_score'] = np.abs((df[feature] - df[feature].mean()) / df[feature].std())

# Any z-score greater than 3 is considered to be an outlier.
# print(df[df['z_score'] > 3])

# Using IQR
Q1 = df[feature].quantile(0.25)
Q3 = df[feature].quantile(0.75)
IQR = Q3 - Q1
# print(df[(df[feature] < (Q1 - 1.5 * IQR)) | (df[feature] > (Q3 + 1.5 * IQR))])

# Using boxplot to visualize the outliers
sns.boxplot(x=df[feature])
plt.show()

In [ ]:
# Plot histogram
plt.hist(df['Temp2m'], bins=30, edgecolor='black')  # Adjust the number of bins as needed
plt.title('Histogram of Temp2m')
plt.xlabel('Temp2m')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Perform the Shapiro-Wilk test
stat, p = shapiro(df['Temp2m'])
print('Statistics=%.3f, p=%.3f' % (stat, p))

# interpret
alpha = 0.05
if p > alpha:
    print('Sample looks Gaussian (fail to reject H0)')
else:
    print('Sample does not look Gaussian (reject H0)')

# Correlation analysis
Below we do correlation analysis on the merged/combined datasets. The findings show correlations within (-0.3, 0.3), hence not very high correlation values, which indicates that it is not worth removing any features.

In [ ]:
path = '../data/NLB/'

# Assuming that 'data.csv' is your data file.
df = pd.read_csv(path + 'nlb_merged_utc_cl.csv', sep='\,') # Change this to the path of your csv file.

# Convert the 'Date' column to datetime format
df['Date'] = pd.to_datetime(df['Date'])

corr_matrix = df.corr()
# Generate a mask for the upper triangle (optional)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# Set up the matplotlib figure
f, ax = plt.subplots(figsize=(11, 9))

# Generate a custom diverging colormap
cmap = sns.diverging_palette(230, 20, as_cmap=True)

# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(corr_matrix, mask=mask, cmap=cmap, vmax=.3, center=0,
            square=True, linewidths=.5, cbar_kws={"shrink": .5})

# Show the plot
plt.show()

# Correlation analysis (cont'd)
Here we run it for individual datasets, and the results are very similar. I've tried manually several different datasets and all correlation values are still within (-0.3, 0.3).

UPDATE: After using the new data that includes high frequency GHI, DHI, and BNI values, as well as solar data (e.g. zenith and apparent zenith), it is evident that apparent zenith and zenith, as well as apparent elevation and elevation, have the same values (in fact, the high correlation prompted me to check the actual .csv files, where i found that they have the exact same values), hence I'm going to be dropping the two 'apparent' features.

In [ ]:
corr_matrix

In [ ]:
import numpy as np
import seaborn as sns
path = '../data/NLB_merged/1-min/moving_averages/'

df = pd.read_csv(path + 'nlb_merged_utc-cl-BaltaSoundCL.csv', sep='\,') # Change this to the path of your csv file.

# Convert the 'Date' column to datetime format
df = df.drop('Date', axis=1)

corr_matrix = df.corr()
# Generate a mask for the upper triangle (optional)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# Set up the matplotlib figure
f, ax = plt.subplots(figsize=(11, 9))

# Generate a custom diverging colormap
cmap = sns.diverging_palette(230, 20, as_cmap=True)

# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(corr_matrix, mask=mask, cmap=cmap, vmax=.3, center=0,
            square=True, linewidths=.5, cbar_kws={"shrink": .5})

# Show the plot
plt.show()

# Dependency between features and target variable
i.e. Mutual information

The mutual_info_classif function computes the mutual information between each feature in X and the target variable y. Mutual information measures the dependency between two random variables: it's equal to zero if and only if two random variables are independent, and higher values indicate higher dependency.

For example, an array such as:

```
[0.11879476 0.00228959 0. 0. 0.00161956 0.38276137 0.38398088 0.16185217]
```

 that you are seeing are the mutual information scores between each feature and the target. You have eight features in total, and the scores correspond to these features. For instance, the first feature has a mutual information score of about 0.11 with the target, the second feature has a score of 0 (implying it's independent of the target), and so on.

***An interesting observation when we run this method to the datasets with 15-min GHI, DHI, and BNI is that for NLB, these three features are not as important as the sunangle, which tends to have significantly higher umutual information score. On the other hand, for TH, we observe a more balanced picture, where GHI, DHI, and BNI have higher scores, but also these scores are similar to some other features.***


**=====What I describe below is not used any more in my analysis======**

The two features with the highest scores (0.38276137 and 0.38398088) are selected by the SelectKBest function since k=2 was specified, indicating that these are the two most informative features according to the mutual information criterion. So from this we can run classification experiments with GHI and DHI only, as these are Feature 5 and 6, which correspond to the array indices 5 and 6 above that had the highest scores of 0.38276137 and 0.38398088 (remember that array index starts from 0).

The 2D array below the mutual information scores represents the selected features for the first five data points (or instances). This is the output of the fit.transform(X) function call which applies the selection to your dataset. In this array, each row corresponds to a data point and each column corresponds to one of the two selected features.

In [ ]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import mutual_info_classif

path = '../data/'

df = pd.read_csv(path + 'BishopRock_CL_I.csv', sep='\,') # Change this to the path of your csv file.

X = df.drop(['Date', 'Value', 'Tagname'], axis=1)
y = df.Value

# Feature extraction
test = SelectKBest(score_func=mutual_info_classif, k=8)

fit = test.fit(X, y)

# Summarize scores
print(fit.scores_)

# Apply the transformation to the dataset
features = fit.transform(X)

# Summarize selected features
print(features[0:7])





# Calculate mutual information
mutual_info = mutual_info_classif(X, y)

# Create dataframe for mutual information for better visualization
mi_df = pd.DataFrame(mutual_info, index=X.columns, columns=['mutual_info']).sort_values(by='mutual_info', ascending=False)

print(mi_df)

# Box plot analysis of datasets

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import fnmatch
import math
from sklearn.feature_selection import mutual_info_classif

path = '../data/TH/'
input = 'input/NLB_merged/1-min/'

filenames = [f for f in os.listdir(path + input) if fnmatch.fnmatch(f, '*.csv')]
filenames = sorted(filenames)

# Dictionary to hold all dataframes
data_dict = {}

for f in filenames:
    print('======================== Dataset: ' + str(f) + '========================')
    data_dict[f] = pd.read_csv(path + input + f, sep='\,')  # Change this to the path of your csv file.

# Variables of interest
variables = ['Sunangle', 'Temp2m', 'Dewpoint2m', 'Pressure', 'Precipitation',
             'GHI_N', 'DHI_N', 'BNI_N',
             'apparent_zenith', 'zenith', 'apparent_elevation', 'elevation', 'azimuth', 'equation_of_time', 'ghi_M',
             'dhi_M', 'dni_M'
             ]

description = []

# Compute the number of rows required for subplots
n = len(variables)
ncols = 2  # Define as many as you need
nrows = math.ceil(n / ncols)

# Create a figure instance
fig, axs = plt.subplots(nrows, ncols, figsize=(15, nrows * 5))

# Flatten to allow easy iteration
axs = axs.flatten()

# First, calculate the mutual information for each dataset and store the results
# in a dictionary, so that we can then print them in the box plot
mutual_info_dict = {}
for f, df in data_dict.items():
    X = df.drop(['Date', 'Value', 'Tagname'], axis=1)
    y = df.Value
    mutual_info = mutual_info_classif(X, y)
    mutual_info_dict[f] = pd.DataFrame(mutual_info, index=X.columns, columns=['mutual_info'])

# Loop over variables of interest
for idx, var in enumerate(variables):
    # Prepare data for boxplots
    data_to_plot = [df[var] for df in data_dict.values()]

    # Create the boxplot
    axs[idx].boxplot(data_to_plot)
    axs[idx].set_xticklabels(filenames, rotation=90)  # Rotates X-axis labels in case they overlap
    axs[idx].set_title(var)

    # Add the mutual information values to the boxplot
    for i, f in enumerate(filenames):
        if var in mutual_info_dict[f].index:
            mi_value = mutual_info_dict[f].loc[var, 'mutual_info']
            axs[idx].text(i+1, axs[idx].get_ylim()[1], f'MI: {mi_value:.2f}', ha='center')

# Delete any unused subplots when number of variables is less than nrows * ncols
if len(variables) < nrows * ncols:
    for idx in range(len(variables), len(axs)):
        fig.delaxes(axs[idx])

plt.tight_layout()
plt.show()

# Compute statistics
for f, df in data_dict.items():
    desc_df = df.describe()
    desc_df['Dataset'] = f
    description.append(desc_df)

description_result = pd.concat(description)
description_reset = description_result.reset_index()
sorted_description = description_reset.sort_values(by=['index', 'Dataset'])
sorted_description


# Statistical tests and analysis
Here I'm comparning the GHI and DHI distributions of the datasets (pairwise comparisons) to see if any of them come from the same continuous distribution. The rationale was that if this is the case, then I could treat these datasets as 'similar', and thus merge them and train them together.

I also do pairwise correlation tests to identify a group of datasets with similar correlation values, which i can then put forward for the merging process.

In [ ]:
from natsort import natsorted, ns
import io
import os
import csv
import fnmatch
import pandas as pd
import matplotlib.pyplot as plt
from itertools import chain, combinations
from scipy import stats

path = '../data/TH/'
input = 'input/'
filenames = [f for f in os.listdir(path + input) if fnmatch.fnmatch(f, '*.csv')]
filenames = sorted(filenames)
datasets_dict = {}
names = []
description = []

for f in filenames:
  data = pd.DataFrame()
  print('======================== Dataset: ' + str(f) + '========================')
  names.append(f)
  df = pd.read_csv(path + input + f, usecols = ['ghi_M', 'dhi_M', 'Value'], sep='\,') # Change this to the path of your csv file.

  desc_df = df.describe()
  desc_df['Dataset'] = f

  description.append(desc_df)

  X = df.copy(deep=True) # Features dataframe
  y = X.pop('Value') # Target/class dataframe
  datasets_dict[str(f)] = X

df_pairs = list(combinations(datasets_dict.keys(), 2))
p_values_GHI = {}
p_values_DHI = {}
correlation_GHI = {}
correlation_DHI = {}
kruskal_GHI = {}
kruskal_DHI = {}
levene_GHI = {}
levene_DHI = {}

for pair in df_pairs:
    # Get dataframes in this pair
    df_a = datasets_dict[pair[0]]
    df_b = datasets_dict[pair[1]]

    # Run the KS test on 'GHI' column
    ks_result_GHI = stats.ks_2samp(df_a['ghi_M'], df_b['ghi_M'])
    # Store p-value
    p_values_GHI[pair] = ks_result_GHI.pvalue

    # Run the KS test on 'DHI' column
    ks_result_DHI = stats.ks_2samp(df_a['dhi_M'], df_b['dhi_M'])
    # Store p-value
    p_values_DHI[pair] = ks_result_DHI.pvalue

    correlation_GHI[pair] = df_a['ghi_M'].corr(df_b['ghi_M'])
    correlation_DHI[pair] = df_a['dhi_M'].corr(df_b['dhi_M'])

# Display p-values
print('KS test P-values for GHI:')
for pair, p_value in p_values_GHI.items():
  if p_value > 0.05: # uncomment if you want to print only the p-values above 5% (none exist!)
    print(f'{pair}: {p_value}')

print('\nKS test P-values for DHI:')
for pair, p_value in p_values_DHI.items():
  if p_value > 0.05:  # uncomment if you want to print only the p-values above 5% (none exist!)
    print(f'{pair}: {p_value}')

print('\nCorrelation for GHI:')
for pair, corr in correlation_GHI.items():
  if corr >= 0.3 or corr <= -0.3:
    print(f'{pair}: {corr}')

print('\nCorrelation for DHI:')
for pair, corr in correlation_DHI.items():
  if corr >= 0.3 or corr <= -0.3:
    print(f'{pair}: {corr}')


def all_combinations(any_list):
    return chain(*map(lambda x: combinations(any_list, x), range(0, len(any_list)+1)))

combos = list(all_combinations(datasets_dict.keys()))

for combo in combos:
    if len(combo) > 1:  # Kruskal-Wallis needs at least two groups
        # For 'GHI'
        data_GHI = [datasets_dict[c]['ghi_M'] for c in combo]
        kruskal_result_GHI = stats.kruskal(*data_GHI)
        kruskal_GHI[combo] = kruskal_result_GHI.pvalue
        levene_result_GHI = stats.levene(*data_GHI)
        levene_GHI[combo] = levene_result_GHI.pvalue

        # For 'DHI'
        data_DHI = [datasets_dict[c]['dhi_M'] for c in combo]
        kruskal_result_DHI = stats.kruskal(*data_DHI)
        kruskal_DHI[combo] = kruskal_result_DHI.pvalue
        levene_result_DHI = stats.levene(*data_DHI)
        levene_DHI[combo] = levene_result_DHI.pvalue

print('P-values for Kruskal-Wallis GHI:')
for combo, p_value in kruskal_GHI.items():
  if p_value > 0.05:
    print(f'{combo}: {p_value}')

print('P-values for Kruskal-Wallis DHI:')
for combo, p_value in kruskal_DHI.items():
  if p_value > 0.05:
    print(f'{combo}: {p_value}')

print('P-values for Levene GHI (values above 0.05 indicate that the variances of the groups are equal [null hypothesis: equality of variances]):')
for combo, p_value in levene_GHI.items():
  if p_value > 0.05:
    print(f'{combo}: {p_value}')

print('P-values for Levene DHI (values above 0.05 indicate that the variances of the groups are equal [null hypothesis: equality of variances]):')
for combo, p_value in levene_DHI.items():
  if p_value > 0.05:
    print(f'{combo}: {p_value}')


In [ ]:
pip install pvlib

In [ ]:
import pandas as pd
import pvlib
from pvlib import location

# Define location
latitude, longitude = 60.76, -0.85  # Balta Sound, Shetland Islands
tz = 'UTC'  # Timezone
loc = location.Location(latitude, longitude, tz=tz)

# Define time range
start = pd.Timestamp('2023-07-18 00:00:00', tz=tz)
end = pd.Timestamp('2023-07-18 23:59:00', tz=tz)
times = pd.date_range(start, end, freq='1min')

# Calculate solar position
solar_position = pvlib.solarposition.get_solarposition(times, latitude, longitude)

# Calculate clear-sky GHI, DNI, and DHI
clearsky = loc.get_clearsky(times, model='ineichen', solar_position=solar_position)

# Combine the solar position and irradiance data
combined = pd.concat([solar_position, clearsky], axis=1)

print(combined)


# Adding additional features
To capture the temporal effects of the existing features on the sensor light, I'm creating moving averages of all features. I'm using a window of 2 steps, which will allow to focus on On/Off pairs and observe how their values are affected over time.

Because we have different types of files for the NLB and TH (including drifted and merged), I need to run this method for each of the above types. Essentially, I need to put in the input folder all relevant files that I want to add the new features, and then these are going to be created in the results folder.

This only needs to happen once - after I've created those files, I can use them as input files for drifting analysis etc.

In [ ]:
path = '../data/TH/'
input = 'input/additional/'
results = 'results/classification2/' # Folder where results are saved
filenames = [f for f in os.listdir(path + input) if fnmatch.fnmatch(f, '*.csv')]
filenames = sorted(filenames)
names = []

for f in filenames:
  data = pd.DataFrame()
  print('======================== Dataset: ' + str(f) + '========================')
  names.append(f)
  df = pd.read_csv(path + input + f, sep='\,') # Change this to the path of your csv file.

  # Removing Date, Value, Tagname, as I don't want to calculate new features (e.g. moving average) for them
  X = df.drop(['Date', 'Value', 'Tagname'], axis=1)

  temp = pd.DataFrame()
  for col_name in X.columns:
    temp[col_name + '_MA'] = X[col_name].rolling(window=2, min_periods=1).mean() # rolling window of 2. Given that there's always alternate states (on/off), it might be useful to have a rolling window of the features' values to see if we can capture how changes in weather conditions affect the sensor's state transitions.
    # I also considered using ratios (next two lines below), but they resulted to a high number of NaN, and as a result I discarded this approach of generating new features.
    #X[col_name + '_ratio'] = X[col_name].pct_change()
    #X[col_name + '_lagged'] = X[col_name] * X['Lagged_values']

  # Concatenate the original (df) dataframe with the newly created temp, which holds the new features. Place them before the last column of df, to ensure that Value (On/Off) remains the last column.
  result = pd.concat([df.iloc[:, :-1], temp, df.iloc[:, -1:]], axis=1)
  result
  result.to_csv(path + results + f, index=False)


In [ ]:
df

# NLB bar graph to show accuracy drops (between 30min and 1min) for 1 hour, 15 min, and 1 miute data

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Data from the image provided by the user
data_1 = {
    'Lighthouses': ['Balta', 'Dunnet', 'Flotta', 'Mousa', 'NRona', 'NRonald', 'Sule', 'Sumburgh'],
    '1-hour': [0.9476, 0.7594, 0.9484, 0.9772, 0.7983, 0.9563, 0.9486, 0.9089],
    '15-minute': [0.9598, 0.7098, 0.9759, 0.9331, 0.5715, 0.9678, 0.9839, 0.9350],
    '1-minute': [0.7651, 0.5692, 0.5949, 0.5756, 0.4945, 0.8117, 0.8391, 0.8130]
}

data_30 = {
    'Lighthouses': ['Balta', 'Dunnet', 'Flotta', 'Mousa', 'NRona', 'NRonald', 'Sule', 'Sumburgh'],
    '1-hour': [0.9560, 0.8587, 0.9181, 0.8800, 0.5244, 0.9534, 0.9463, 0.9145],
    '15-minute': [0.9530, 0.8415, 0.7457, 0.6420, 0.5029, 0.9770, 0.9424, 0.9327],
    '1-minute': [0.6319, 0.7024, 0.5160, 0.4904, 0.5119, 0.6271, 0.6803, 0.6282],
}

# Calculate the differences
difference_data = {
    'Lighthouses': data_1['Lighthouses'],  # Keep the Lighthouses list
    '1-hour': [a - b for a, b in zip(data_30['1-hour'], data_1['1-hour'])],
    '15-minute': [a - b for a, b in zip(data_30['15-minute'], data_1['15-minute'])],
    '1-minute': [a - b for a, b in zip(data_30['1-minute'], data_1['1-minute'])]
}

# Convert the data to a Pandas DataFrame
df = pd.DataFrame(difference_data)

# Set the Lighthouses as the index
df.set_index('Lighthouses', inplace=True)

# Plotting
fig, ax = plt.subplots(figsize=(14, 8))

# Plot for each time data
df['1-hour'].plot(kind='bar', color='skyblue', ax=ax, width=0.25, position=2, label='1-hour Data')
df['15-minute'].plot(kind='bar', color='orange', ax=ax, width=0.25, position=1, label='15-minute Data')
df['1-minute'].plot(kind='bar', color='green', ax=ax, width=0.25, position=0, label='1-minute Data')

# Adjusting the x-axis limits
ax.set_xlim(-1, len(df.index))  # Setting the x-axis limits

# Setting the labels and titles
ax.set_ylabel('Difference in NLB Accuracy')
ax.set_xlabel('Lighthouses')
ax.set_title('Difference in NLB Accuracy for Different Data Intervals')
ax.legend()

# Show the plot
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# TH bar graph to show accuracy drops (between 30min and 1min) for 1 hour, 15 min, and 1 miute data

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Data from the image provided by the user
data_1 = {
    'Lighthouses': ['Bishop', 'Eddy', 'Godrevy', 'Lizard', 'Longships', 'Trevose', 'Wolf'],
    '1-hour': [0.8282, 0.8209, 0.9408, 0.8264, 0.8349, 0.8271, 0.845],
    '15-minute': [0.8446, 0.7948, 0.9282, 0.8215, 0.8363, 0.8352, 0.845],
    '1-minute': [0.6342, 0.5738, 0.644, 0.6131, 0.6483, 0.6399, 0.6388]
}

data_30 = {
    'Lighthouses': ['Balta', 'Dunnet', 'Flotta', 'Mousa', 'NRona', 'NRonald', 'Sule', 'Sumburgh'],
    '1-hour': [0.7628, 0.8334, 0.8999, 0.5002, 0.7958, 0.7614, 0.798],
    '15-minute': [0.6178, 0.8429, 0.8905, 0.6069, 0.7368, 0.7094, 0.795],
    '1-minute': [0.4845, 0.6295, 0.5391, 0.4799, 0.5411, 0.5045, 0.5819],
}

# Calculate the differences
difference_data = {
    'Lighthouses': data_1['Lighthouses'],  # Keep the Lighthouses list
    '1-hour': [a - b for a, b in zip(data_30['1-hour'], data_1['1-hour'])],
    '15-minute': [a - b for a, b in zip(data_30['15-minute'], data_1['15-minute'])],
    '1-minute': [a - b for a, b in zip(data_30['1-minute'], data_1['1-minute'])]
}

# Convert the data to a Pandas DataFrame
df = pd.DataFrame(difference_data)

# Set the Lighthouses as the index
df.set_index('Lighthouses', inplace=True)

# Plotting
fig, ax = plt.subplots(figsize=(14, 8))

# Plot for each time data
df['1-hour'].plot(kind='bar', color='skyblue', ax=ax, width=0.25, position=2, label='1-hour Data')
df['15-minute'].plot(kind='bar', color='orange', ax=ax, width=0.25, position=1, label='15-minute Data')
df['1-minute'].plot(kind='bar', color='green', ax=ax, width=0.25, position=0, label='1-minute Data')

# Adjusting the x-axis limits
ax.set_xlim(-1, len(df.index))  # Setting the x-axis limits

# Setting the labels and titles
ax.set_ylabel('Difference in TH Accuracy')
ax.set_xlabel('Lighthouses')
ax.set_title('Difference in TH Accuracy for Different Data Intervals')
ax.legend()

# Show the plot
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# Bar graph for NLB features occurrence

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

# Extracted data from the provided spreadsheet
features = [
    "Sunangle", "Temp2m", "Sunangle", "Temp2m",
    "Temp2m", "Dewpoint2m", "Temp2m", "Dewpoint2m",
    "Dewpoint2m", "Precipitation", "DHI_N", "Precipitation",
    "Precipitation", "GHI_N", "Zenith", "DHI_N",
    "Elevation", "Apparent Elevation", "Apparent Elevation", "Apparent Elevation",
    "Azimuth", "GHI_M", "Azimuth", "Elevation",
    "Equation of time", "DHI_M", "DNI_M", "SunangleMA",
    "GHI_M", "SunangleMA", "SunangleMA", "Equation of timeMA",
    "", "DNI_M_MA", "GHI_N_MA", "DNI_M_MA",
    "", "", "ZenithMA", "",
    "", "", "Elevation_MA", "",
    "", "", "GHI_M_MA", "",
    "", "", "DNI_M_MA", ""
]

# Remove empty strings from the data
cleaned_features = [feature for feature in features if feature]

# Count the occurrences of each feature
feature_counts = Counter(cleaned_features)

# Plotting the bar graph
plt.figure(figsize=(10, 8))
plt.bar(feature_counts.keys(), feature_counts.values(), color='teal')

# Rotate labels for better readability
plt.xticks(rotation=90)

# Add labels and title
plt.xlabel('Features')
plt.ylabel('Occurrences')
plt.title('Number of Occurrences of Each NLB Feature')

# Show the plot
plt.tight_layout()
plt.show()


# Bar graph for TH features occurrence

In [ ]:
# Re-importing necessary libraries and re-defining the data to plot the graph
from collections import Counter
import matplotlib.pyplot as plt

# Data extracted from the provided spreadsheet information
features = [
    "Sunangle", "Sunangle", "Temp2m", "Precipitation", "Dewpoint2m",
    "Temp2m", "Dewpoint2m", "Dewpoint2m", "Apparent Zenith", "Precipitation",
    "Dewpoint2m", "Precipitation", "GHI_N", "Apparent Elevation", "GHI_N",
    "Precipitation", "GHI_N", "DHI_N", "Elevation", "DHI_N",
    "BNI_N", "DHI_N", "Apparent Zenith", "Azimuth", "Apparent Zenith",
    "Apparent Elevation", "BNI_N", "Apparent Elevation", "GHI_M", "Zenith",
    "Elevation", "Apparent Zenith", "Azimuth", "DNI_M", "Azimuth",
    "Azimuth", "Zenith", "Equation of time", "SunangleMA", "GHI_M",
    "Equation of time", "Apparent Elevation", "DNI_M", "GHI_N_MA", "DHI_M",
    "SunangleMA", "Elevation", "SunangleMA", "DHI_N_MA", "DNI_M",
    "GHI_M_MA", "Azimuth", "DHI_N_MA", "GHI_M_MA", "SunangleMA",
    "DHI_M_MA", "Equation of time", "Equation of timeMA", "DNI_M_MA", "ElevationMA",
    "DNI_M_MA", "ZenithMA", "GHI_M_MA", "", "DHI_M_MA",
    "Equation of timeMA", "DNI_M_MA", "", "DNI_M_MA",
    "DHI_M_MA", "", "", ""
]

# Remove empty strings from the data
cleaned_features = [feature for feature in features if feature]

# Count the occurrences of each feature
feature_counts = Counter(cleaned_features)

# Plotting the bar graph
plt.figure(figsize=(10, 8))
plt.bar(feature_counts.keys(), feature_counts.values(), color='teal')

# Rotate labels for better readability
plt.xticks(rotation=90)

# Add labels and title
plt.xlabel('Features')
plt.ylabel('Occurrences')
plt.title('Number of Occurrences of Each TH Feature')

# Show the plot
plt.tight_layout()
plt.show()


# Example of a 2 objective plot for Pareto front

In [ ]:
# To plot the efficient frontier (Pareto front), we first need to find the Pareto optimal points
def identify_pareto(scores):
    """
    Identify the Pareto optimal points
    Input:
    - scores: An array where each row contains the scores for a particular solution, and each column a different objective
    Output:
    - A boolean array indicating whether each point is Pareto optimal
    """
    is_efficient = np.ones(scores.shape[0], dtype=bool)
    for i, c in enumerate(scores):
        if is_efficient[i]:
            is_efficient[is_efficient] = np.any(scores[is_efficient] < c, axis=1)  # Keep any point with a lower score
            is_efficient[i] = True  # And keep self
    return is_efficient

# Combine the objectives into a single array
scores = np.vstack((objective_1, objective_2)).T

# Identify Pareto optimal points
pareto_optimal = identify_pareto(scores)

# Plotting the data along with the efficient frontier
plt.figure(figsize=(10, 6))
plt.scatter(objective_1[~pareto_optimal], objective_2[~pareto_optimal], color='blue', label='Non-Pareto Optimal')
plt.scatter(objective_1[pareto_optimal], objective_2[pareto_optimal], color='red', label='Pareto Optimal')

# Highlight the efficient frontier
sorted_pareto_points = scores[pareto_optimal]
sorted_pareto_points = sorted_pareto_points[np.argsort(sorted_pareto_points[:, 0])]
plt.plot(sorted_pareto_points[:, 0], sorted_pareto_points[:, 1], color='green', label='Pareto Front')

# Enhancing the plot
# plt.title("Pareto Front and Efficient Frontier for a Two-Objective Minimisation Problem")
plt.xlabel("Objective 1")
plt.ylabel("Objective 2")
plt.legend()
plt.grid(True)
plt.show()
